In [ ]:
import pandas as pd
import numpy as np
import sys 
import os


In [ ]:
import pickle

# Caminho para o ficheiro exportado (ajuste se guardou noutra pasta)
pkl_path = '../DynamicGraphPkls/dynamic_graphs_output_kendall_0.8_Window7_Step1.pkl'

print(f"A carregar grafos e features pré-computados de {pkl_path}...")
with open(pkl_path, 'rb') as f:
    graph_data = pickle.load(f)

# Extrair todas as variáveis de volta para o ambiente do notebook
graphs = graph_data['graphs']
sim_dfs = graph_data['sim_dfs']
df_pivots = graph_data['df_pivots']
window_info = graph_data['window_info']
dynamic_graph_features = graph_data['dynamic_graph_features']


SIMILARITY_METHOD = graph_data['similarity_method']
WINDOW_SIZE = graph_data['window_size']
STEP_SIZE = graph_data['step_size']
print(f"Sucesso! Foram carregadas as informações de {len(graphs)} janelas temporais.")
print(f"Dimensão das features do nó na janela 1: {dynamic_graph_features[0].shape}")

In [ ]:
'''
from DynamicSimilarities.plot import save_dynamic_graph_plots
SIMILARITY_METHOD = 'kendall'
WINDOW_SIZE = 7
STEP_SIZE = 1
save_dynamic_graph_plots(graphs, SIMILARITY_METHOD, node_categories=None, window_size=WINDOW_SIZE, step_size=STEP_SIZE)
'''

In [ ]:
import networkx as nx
from collections import Counter
from DynamicSimilarities.plot import save_graph_plot

# 1. Count how many times each edge appears across all temporal graphs
edge_counts = Counter()
total_graphs = len(graphs)

for g in graphs:
    # Ensure edge tuples are sorted so (A, B) and (B, A) are counted as the same edge
    for u, v in g.edges():
        edge = tuple(sorted((str(u), str(v))))
        edge_counts[edge] += 1

# 2. Create a DataFrame to analyze the persistence of each relationship
persistent_edges = pd.DataFrame(
    [(u, v, count, count / total_graphs * 100) for (u, v), count in edge_counts.items()],
    columns=['Node1', 'Node2', 'Occurrences', 'Persistence (%)']
)

# Sort by persistence (descending) to find the most stable relationships
persistent_edges = persistent_edges.sort_values(by='Persistence (%)', ascending=False).reset_index(drop=True)

print(f"Analysis over {total_graphs} temporal graphs.")
print(f"Total unique edges that appeared at least once: {len(persistent_edges)}")
print("\nTop 20 most stable product relationships:")
display(persistent_edges.head(20))

# 3. Optional: Create a "Consensus Graph" with edges present in at least X% of the time (e.g., 50%)
THRESHOLD_PERCENT = 50.0
stable_graph = nx.Graph()

# Add only the edges that meet the threshold
stable_edges = persistent_edges[persistent_edges['Persistence (%)'] >= THRESHOLD_PERCENT]
for _, row in stable_edges.iterrows():
    stable_graph.add_edge(row['Node1'], row['Node2'], weight=row['Persistence (%)'])

print(f"\nCreated a Consensus Graph with {stable_graph.number_of_edges()} edges (Persistence >= {THRESHOLD_PERCENT}%)")

# Optional: Save the consensus graph plot
save_graph_plot(
    G=stable_graph, 
    strategy_name=f"Consensus_Graph_Persistence_{SIMILARITY_METHOD}_{int(THRESHOLD_PERCENT)}",
    output_folder="consensus_graph_plots",
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Let's say we define "most persistent" as edges that exist in at least 50% of the windows
PERSISTENCE_THRESHOLD = 0.5 
total_windows = len(graphs)
min_occurrences = total_windows * PERSISTENCE_THRESHOLD

# 1. Gather all occurrences per edge over time
edge_presence = {}

for w_idx, g_data in enumerate(graphs):
    # Handle dict (adj_list) vs NetworkX format
    if isinstance(g_data, dict) and "adj_list" in g_data:
        edges = []
        for node, neighbors in g_data["adj_list"].items():
            for neighbor_info in neighbors:
                edges.append((str(node), str(neighbor_info["node"])))
    else:
        edges = g_data.edges()
        
    for u, v in edges:
        edge = tuple(sorted((str(u), str(v))))
        if edge not in edge_presence:
            edge_presence[edge] = [0] * total_windows
        edge_presence[edge][w_idx] = 1

# 2. Filter edges to only those that cross the persistence threshold
persistent_edges = {e: pres for e, pres in edge_presence.items() if sum(pres) >= min_occurrences}

# Option: Sort these specific persistent edges by total count so the most stable are on top
sorted_filtered_edges = sorted(persistent_edges.keys(), key=lambda e: sum(persistent_edges[e]), reverse=True)

if not sorted_filtered_edges:
    print(f"No edges matched the {PERSISTENCE_THRESHOLD*100}% threshold. Adjust your threshold.")
else:
    presence_matrix = [persistent_edges[e] for e in sorted_filtered_edges]
    edge_labels = [f"{e[0]} - {e[1]}" for e in sorted_filtered_edges]

    # 3. Plot the limited heatmap
    plt.figure(figsize=(16, max(4, len(sorted_filtered_edges) * 0.2))) # Dynamically scale height based on number of edges

    ax = sns.heatmap(
        presence_matrix, 
        cmap="Blues", 
        cbar=False, 
        xticklabels=max(1, total_windows // 20),  # Scale x ticks
        yticklabels=True
    )

    plt.title(f"Edge Persistence Over Time Heatmap (Threshold: {PERSISTENCE_THRESHOLD*100}%) - Total Edges: {len(sorted_filtered_edges)}")
    plt.xlabel("Time Windows (Indices)")
    plt.ylabel("Highly Persistent Edges (Item Pairs)")

    ax.set_yticks(np.arange(len(edge_labels)) + 0.5)
    ax.set_yticklabels(edge_labels, rotation=0, fontsize=8)

    plt.tight_layout()
    plt.show()